In [ ]:
from google.colab import files

uploaded = files.upload()

Saving S01R01.txt to S01R01.txt
Saving S01R02.txt to S01R02.txt
Saving S02R01.txt to S02R01.txt
Saving S02R02.txt to S02R02.txt
Saving S03R01.txt to S03R01.txt
Saving S03R02.txt to S03R02.txt
Saving S03R03.txt to S03R03.txt
Saving S04R01.txt to S04R01.txt
Saving S05R01.txt to S05R01.txt
Saving S05R02.txt to S05R02.txt
Saving S06R01.txt to S06R01.txt
Saving S06R02.txt to S06R02.txt
Saving S07R01.txt to S07R01.txt
Saving S07R02.txt to S07R02.txt
Saving S08R01.txt to S08R01.txt
Saving S09R01.txt to S09R01.txt
Saving S10R01.txt to S10R01.txt


In [ ]:
import glob
import os
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut

txt_files = glob.glob("/content/S*R*.txt")

print("Found", len(txt_files), "recordings")
print(txt_files[:5])

Found 17 recordings
['/content/S07R01.txt', '/content/S06R01.txt', '/content/S08R01.txt', '/content/S06R02.txt', '/content/S01R01.txt']


In [ ]:
FS, WIN, HOP = 64, 256, 32  # 4-second window, 0.5-second hop

LOCO = list(range(2, 12))    # 0.5-2.75 Hz
FREEZE = list(range(12, 33)) # 3-8 Hz

HANN = np.hanning(WIN)
EPS = 1e-6

In [ ]:
def features(seg):
    """Extract freezing-band and locomotion-band log power."""
    seg = (seg - seg.mean(axis=0)) * HANN[:, None]

    P = (np.abs(np.fft.rfft(seg, axis=0)) ** 2 / WIN).sum(axis=1)

    freeze_power = np.log10(P[FREEZE].sum() + EPS)
    loco_power = np.log10(P[LOCO].sum() + EPS)

    return [freeze_power, loco_power]

In [ ]:
files = sorted(
    glob.glob("/content/**/S*R*.txt", recursive=True)
)

if not files:
    raise FileNotFoundError(
        "No recordings found! Check that your .txt files "
        "are uploaded and located under /content."
    )

print(f"Found {len(files)} recordings.")

X, y, g = [], [], []

for f in files:
    # Expected filenames, e.g. S01R01.txt
    filename = os.path.basename(f)
    subj = int(filename[1:3])

    d = np.loadtxt(f)

    # Ankle accelerometer, milligravity -> g
    acc = d[:, 1:4] / 1000.0

    # Label column
    lab = d[:, 10].astype(int)

    # Sliding windows
    for s in range(0, len(acc) - WIN, HOP):
        l = lab[s:s + WIN]

        # Retain windows where at least 90% of labels
        # belong to the experiment (labels > 0).
        if (l > 0).mean() < 0.9:
            continue

        X.append(features(acc[s:s + WIN]))

        # Freeze if at least 50% of window labels are 2.
        y.append(int((l == 2).mean() >= 0.5))

        g.append(subj)

X = np.asarray(X)
y = np.asarray(y)
g = np.asarray(g)

if len(y) == 0:
    raise ValueError(
        "No valid windows found. Check the dataset format "
        "and label column."
    )

if len(np.unique(y)) < 2:
    raise ValueError(
        "Only one class was found. Check the labels and "
        "window filtering."
    )

print(f"Extracted {len(y)} windows.")
print(f"Freeze windows: {y.sum()}")
print(f"Non-freeze windows: {(y == 0).sum()}")
print(f"Subjects: {np.unique(g)}")


Found 17 recordings.
Extracted 35405 windows.
Freeze windows: 3384
Non-freeze windows: 32021
Subjects: [ 1  2  3  4  5  6  7  8  9 10]


In [ ]:
logo = LeaveOneGroupOut()

print("\nLeave-one-subject-out evaluation:")
print("-" * 55)

for tr, te in logo.split(X, y, groups=g):
    # Fit scaler ONLY on training subjects
    sc = StandardScaler().fit(X[tr])

    clf = LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    )
    clf.fit(sc.transform(X[tr]), y[tr])

    p = clf.predict(sc.transform(X[te]))

    tp = ((p == 1) & (y[te] == 1)).sum()
    fn = ((p == 0) & (y[te] == 1)).sum()
    tn = ((p == 0) & (y[te] == 0)).sum()
    fp = ((p == 1) & (y[te] == 0)).sum()

    sens = tp / (tp + fn) if tp + fn else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")

    subject = g[te][0]

    print(
        f"S{subject:02d}  "
        f"sensitivity={sens:.2f}  "
        f"specificity={spec:.2f}  "
        f"(freeze windows: {tp + fn})"
    )


Leave-one-subject-out evaluation:
-------------------------------------------------------
S01  sensitivity=0.99  specificity=0.41  (freeze windows: 183)
S02  sensitivity=0.88  specificity=0.83  (freeze windows: 357)
S03  sensitivity=0.93  specificity=0.73  (freeze windows: 566)
S04  sensitivity=nan  specificity=0.92  (freeze windows: 0)
S05  sensitivity=0.63  specificity=0.85  (freeze windows: 956)
S06  sensitivity=0.55  specificity=0.85  (freeze windows: 253)
S07  sensitivity=0.79  specificity=0.87  (freeze windows: 141)
S08  sensitivity=0.83  specificity=0.50  (freeze windows: 405)
S09  sensitivity=0.74  specificity=0.83  (freeze windows: 523)
S10  sensitivity=nan  specificity=0.91  (freeze windows: 0)


In [ ]:
sc = StandardScaler().fit(X)

clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)
clf.fit(sc.transform(X), y)

# Convert standardized-feature coefficients into raw-feature
# coefficients, so the embedded code needs no scaler.
w = clf.coef_[0] / sc.scale_
b = clf.intercept_[0] - np.sum(
    clf.coef_[0] * sc.mean_ / sc.scale_
)

In [ ]:
output_path = "/content/fog_model.h"

with open(output_path, "w") as fh:
    fh.write("#pragma once\n\n")
    fh.write(f"#define FOG_FS {FS}\n")
    fh.write(f"#define FOG_WIN {WIN}\n")
    fh.write(f"#define FOG_HOP {HOP}\n\n")

    fh.write(f"static const float FOG_W_FREEZE = {w[0]:.8f}f;\n")
    fh.write(f"static const float FOG_W_LOCO = {w[1]:.8f}f;\n")
    fh.write(f"static const float FOG_BIAS = {b:.8f}f;\n")

print(f"\nModel saved to: {output_path}")
print(f"Freeze-band weight: {w[0]:.6f}")
print(f"Locomotion-band weight: {w[1]:.6f}")
print(f"Bias: {b:.6f}")


Model saved to: /content/fog_model.h
Freeze-band weight: 4.381337
Locomotion-band weight: -3.551720
Bias: -1.085475


In [ ]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
Event-level check of the FOG detector on Daphnet (leave-one-subject-out).

Run in the same Colab session as fog_ml_model.ipynb (files in /content).
Reproduces the firmware logic: log-band-power features -> logistic score z
-> threshold -> 'confirm' consecutive positives -> FREEZE START/END.

Outputs per subject and overall:
  - window-level sensitivity / specificity
  - episode detection rate and detection latency (onset -> detector START)
  - false alarms per 10 min of non-freeze walking
  - freeze-episode duration stats (to judge what latency is acceptable)
"""
import glob, os
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

FS, WIN, HOP = 64, 256, 32
LOCO, FREEZE = range(2, 12), range(12, 33)
HANN = np.hanning(WIN)
EPS = 1e-6
CONFIRM = 2          # must match FogDetector.begin(..., confirm)
THR = 0.0            # must match FogDetector.begin(..., threshold)
TOL_S = 2.0          # a START up to 2 s after an episode ends is not a false alarm


def feats(seg):
    seg = (seg - seg.mean(axis=0)) * HANN[:, None]
    P = (np.abs(np.fft.rfft(seg, axis=0)) ** 2 / WIN).sum(axis=1)
    return [np.log10(P[list(FREEZE)].sum() + EPS), np.log10(P[list(LOCO)].sum() + EPS)]


def load(pattern="/content/**/S*R*.txt"):
    recs = []
    for f in sorted(glob.glob(pattern, recursive=True)):
        d = np.loadtxt(f)
        acc, lab = d[:, 1:4] / 1000.0, d[:, 10].astype(int)   # ankle accel (g), label
        starts = list(range(0, len(acc) - WIN, HOP))
        X = np.array([feats(acc[s:s + WIN]) for s in starts])
        end_t = np.array([(s + WIN) / FS for s in starts])     # time detector 'knows' the window
        valid = np.array([(lab[s:s + WIN] > 0).mean() >= 0.9 for s in starts])
        y = np.array([int((lab[s:s + WIN] == 2).mean() >= 0.5) for s in starts])
        recs.append(dict(subj=int(os.path.basename(f)[1:3]), X=X, t=end_t,
                         valid=valid, y=y, lab=lab))
    return recs


def episodes(lab):
    """(onset_s, end_s) of each contiguous run of label==2."""
    fz = np.r_[0, (lab == 2).astype(int), 0]
    on, off = np.where(np.diff(fz) == 1)[0], np.where(np.diff(fz) == -1)[0]
    return [(a / FS, b / FS) for a, b in zip(on, off)]


def run_state_machine(z, thr=THR, confirm=CONFIRM):
    pos = neg = 0
    state, states, starts = False, [], []
    for i, zi in enumerate(z):
        if zi > thr: pos, neg = pos + 1, 0
        else:        neg, pos = neg + 1, 0
        if not state and pos >= confirm: state = True;  starts.append(i)
        if state and neg >= confirm:     state = False
        states.append(state)
    return np.array(states), starts


recs = load()
subjects = sorted({r["subj"] for r in recs})
lat_all, det, tot, fa, nonfz_min, durs = [], 0, 0, 0, 0.0, []

for s in subjects:
    tr = [r for r in recs if r["subj"] != s]
    te = [r for r in recs if r["subj"] == s]
    Xtr = np.vstack([r["X"][r["valid"]] for r in tr])
    ytr = np.concatenate([r["y"][r["valid"]] for r in tr])
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(class_weight="balanced", max_iter=1000).fit(sc.transform(Xtr), ytr)
    w = clf.coef_[0] / sc.scale_
    b = clf.intercept_[0] - np.sum(clf.coef_[0] * sc.mean_ / sc.scale_)

    s_lat, s_det, s_tot, s_fa, s_min = [], 0, 0, 0, 0.0
    tp = fn = tn = fp = 0
    for r in te:
        z = r["X"] @ w + b
        p, y, v = (z > THR).astype(int), r["y"], r["valid"]
        tp += ((p == 1) & (y == 1) & v).sum(); fn += ((p == 0) & (y == 1) & v).sum()
        tn += ((p == 0) & (y == 0) & v).sum(); fp += ((p == 1) & (y == 0) & v).sum()

        states, starts = run_state_machine(z)
        eps = episodes(r["lab"])
        for on, off in eps:
            durs.append(off - on)
            s_tot += 1
            hit = np.where(states & (r["t"] >= on) & (r["t"] <= off + TOL_S))[0]
            if len(hit):
                s_det += 1
                s_lat.append(max(0.0, r["t"][hit[0]] - on))
        for i in starts:
            t0 = r["t"][i]
            if not any(on <= t0 <= off + TOL_S for on, off in eps):
                s_fa += 1
        s_min += ((r["lab"] == 1).sum() / FS) / 60.0   # minutes of labelled non-freeze

    sens = tp / (tp + fn) if tp + fn else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")
    print(f"S{s:02d} win-sens={sens:.2f} win-spec={spec:.3f} | episodes {s_det}/{s_tot} "
          f"| median latency {np.median(s_lat) if s_lat else float('nan'):.2f}s "
          f"| false alarms/10min {10 * s_fa / s_min if s_min else float('nan'):.2f}")
    lat_all += s_lat; det += s_det; tot += s_tot; fa += s_fa; nonfz_min += s_min

print("\nOVERALL")
print(f"episodes detected: {det}/{tot} ({100 * det / max(tot, 1):.0f}%)")
if lat_all:
    print(f"latency s: median {np.median(lat_all):.2f}, 90th pct {np.percentile(lat_all, 90):.2f}, "
          f"fraction < 1 s: {np.mean(np.array(lat_all) < 1):.2f}")
print(f"false alarms per 10 min non-freeze: {10 * fa / nonfz_min:.2f}")
if durs:
    print(f"episode duration s: median {np.median(durs):.1f}, 25th pct {np.percentile(durs, 25):.1f}, "
          f"fraction shorter than 3 s: {np.mean(np.array(durs) < 3):.2f}")

S01 win-sens=0.99 win-spec=0.414 | episodes 23/23 | median latency 0.33s | false alarms/10min 74.19
S02 win-sens=0.88 win-spec=0.830 | episodes 24/24 | median latency 1.09s | false alarms/10min 50.58
S03 win-sens=0.93 win-spec=0.733 | episodes 49/49 | median latency 0.28s | false alarms/10min 35.16
S04 win-sens=nan win-spec=0.921 | episodes 0/0 | median latency nans | false alarms/10min 22.03
S05 win-sens=0.63 win-spec=0.855 | episodes 62/66 | median latency 1.38s | false alarms/10min 30.83
S06 win-sens=0.55 win-spec=0.851 | episodes 8/10 | median latency 1.16s | false alarms/10min 41.00
S07 win-sens=0.79 win-spec=0.872 | episodes 23/24 | median latency 0.62s | false alarms/10min 27.48
S08 win-sens=0.83 win-spec=0.497 | episodes 14/14 | median latency 0.20s | false alarms/10min 95.93
S09 win-sens=0.74 win-spec=0.829 | episodes 27/27 | median latency 0.47s | false alarms/10min 31.43
S10 win-sens=nan win-spec=0.914 | episodes 0/0 | median latency nans | false alarms/10min 21.79

OVERALL
